# Six organ-specific vasculature-to-FTU diagrams (Vega)

This notebook reads only the six curated organ CSVs. For every route, negative
`PathStep` rows are arterial flow toward one explicit `FTUID`, and positive rows
are venous return from that FTU toward Heart. The layout is fixed as **Heart (far
left) → organ vasculature (middle) → one black-square endpoint per unique FTUID
(far right)**.

The helper uses side plus `PathVesselID` for vessel identity and a stable
side/name fallback only when the ID is missing. Every rendered vessel is
provenanced to the selected organ CSV; no global table or cached topology is used.


## 1. Paths and Vega helper


In [119]:
from pathlib import Path
import importlib
import json

import pandas as pd
import vl_convert as vlc

import organ_vasculature_vega as organ_vega

# Run All must use the helper currently saved beside this notebook, even when
# this kernel previously imported an older layout version.
organ_vega = importlib.reload(organ_vega)
ORGAN_ORDER = organ_vega.ORGAN_ORDER
build_organ_model = organ_vega.build_organ_model
format_validation = organ_vega.format_validation
make_vega_spec = organ_vega.make_vega_spec

PROJECT_DIR = Path(
    '/Users/zeynepamac/Desktop/Butterfly_v2.5_Update/'
    'ftus_connected_by_vasculature_per_organ'
)
CSV_DIR = PROJECT_DIR / 'organs_with_ftu_6_csv'
OUTPUT_DIR = PROJECT_DIR

assert CSV_DIR.is_dir(), CSV_DIR
print('CSV folder:', CSV_DIR)
print('Output folder:', OUTPUT_DIR)


CSV folder: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftus_connected_by_vasculature_per_organ/organs_with_ftu_6_csv
Output folder: /Users/zeynepamac/Desktop/Butterfly_v2.5_Update/ftus_connected_by_vasculature_per_organ


## 2. Build and print validation before rendering


In [120]:
models = {}
for organ_key in ORGAN_ORDER:
    model = build_organ_model(CSV_DIR, organ_key)
    models[organ_key] = model
    print(format_validation(model))
    route_summary = pd.DataFrame(model['route_summaries'])
    print(route_summary.to_string(index=False))
    print()


KIDNEY | source=kidney_ftu_vascular_data.csv
  unique FTUs=8 | FTUID count=8 | arterial nodes/edges=17/34 | venous nodes/edges=14/32
  duplicate FTU endpoints=0 | missing path IDs=1 | unsupported/cross-organ vessels=0
  FTUs: cortical collecting duct [UBERON:0004203]; descending limb of loop of Henle [UBERON:0001289]; inner medullary collecting duct [UBERON:0004205]; loop of Henle ascending limb thin segment [UBERON:0004193]; nephron [UBERON:0001285]; outer medullary collecting duct [UBERON:0004204]; renal corpuscle [UBERON:0001229]; thick ascending limb of loop of Henle [UBERON:0001291]
                                      FTU          FTUID                 vascular_target  min_PathStep  max_PathStep  arterial_path_nodes  venous_path_nodes
                          renal corpuscle UBERON:0001229            glomerular capillary           -13            12                   13                 12
                          renal corpuscle UBERON:0001229        renal afferent arteriole   

## 3. Required integrity checks


In [121]:
for organ_key, model in models.items():
    validation = model['validation']
    ftu_nodes = [node for node in model['nodes'] if node['type'] == 'ftu']
    vessel_nodes = [node for node in model['nodes'] if node['type'] in {'artery', 'vein'}]
    assert len(ftu_nodes) == validation['unique_ftus']
    assert len(ftu_nodes) == validation['ftuid_count']
    assert validation['duplicate_ftu_endpoint_count'] == 0
    assert validation['unsupported_or_cross_organ_vessels'] == 0
    heart = [node for node in model['nodes'] if node['type'] == 'heart']
    assert len(heart) == 1
    assert len({node['x'] for node in ftu_nodes}) == 1
    assert min(node['x'] for node in ftu_nodes) > max(node['x'] for node in vessel_nodes)
    assert heart[0]['x'] < min(node['x'] for node in vessel_nodes)
print('All six models passed FTUID uniqueness, provenance, and left/right layout checks.')


All six models passed FTUID uniqueness, provenance, and left/right layout checks.


## 4. Render six Vega JSON and SVG pairs


In [122]:
rendered = {}
for organ_key in ORGAN_ORDER:
    model = models[organ_key]
    spec = make_vega_spec(model)
    stem = f'{organ_key}_vasculature_ftus_vega'
    json_path = OUTPUT_DIR / f'{stem}.json'
    svg_path = OUTPUT_DIR / f'{stem}.svg'
    json_path.write_text(
        json.dumps(spec, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
    )
    svg_text = vlc.vega_to_svg(vg_spec=spec)
    svg_path.write_text(svg_text, encoding='utf-8')
    assert '<svg' in svg_text and 'Heart' in svg_text
    rendered[organ_key] = {'json': json_path, 'svg': svg_path}
    print(f'{organ_key:16s} -> {svg_path.name}; {json_path.name}')


kidney           -> kidney_vasculature_ftus_vega.svg; kidney_vasculature_ftus_vega.json
liver            -> liver_vasculature_ftus_vega.svg; liver_vasculature_ftus_vega.json
large_intestine  -> large_intestine_vasculature_ftus_vega.svg; large_intestine_vasculature_ftus_vega.json
small_intestine  -> small_intestine_vasculature_ftus_vega.svg; small_intestine_vasculature_ftus_vega.json
mouth            -> mouth_vasculature_ftus_vega.svg; mouth_vasculature_ftus_vega.json
prostate         -> prostate_vasculature_ftus_vega.svg; prostate_vasculature_ftus_vega.json


## 5. Final endpoint and file verification


In [123]:
for organ_key, files in rendered.items():
    spec = json.loads(files['json'].read_text(encoding='utf-8'))
    ftu_values = next(data['values'] for data in spec['data'] if data['name'] == 'ftus')
    explicit_ids = {item['id'] for item in models[organ_key]['validation']['ftus']}
    assert {node['ontology_id'] for node in ftu_values} == explicit_ids
    assert len(ftu_values) == len(explicit_ids)
    assert files['svg'].is_file() and files['json'].is_file()
print('Complete: six SVGs, six Vega JSON files, and exactly 15 unique FTUID endpoints.')


Complete: six SVGs, six Vega JSON files, and exactly 15 unique FTUID endpoints.
